<img src="https://raw.githubusercontent.com/AliAkrami1375/dcho/main/assets/logo.png" width="90">

# dcho — training on Colab

Persian text-to-speech. Trains here, checkpoints to the Hugging Face Hub.

**Read this first.** Colab sessions end — after about 12 hours on the free
tier, sooner if the tab closes or the runtime is reclaimed. This notebook is
built around that rather than against it: it pushes a checkpoint to the Hub
every 2,000 steps and, on the next run, resumes from exactly where it
stopped. Losing a session costs the minutes since the last checkpoint, not
the run.

**Two things to set before running:**

1. `Runtime → Change runtime type → GPU`
2. The key icon 🔑 in the left sidebar → add a secret named `HF_TOKEN`
   holding a Hugging Face token with write access

## 1 · What hardware did we get?

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

# Rough throughput relative to an L4, used further down to project how long
# a phase will take on whatever Colab handed us this session.
name = torch.cuda.get_device_name()
RELATIVE = {"T4": 0.37, "L4": 1.00, "A100": 2.50, "V100": 0.75, "P100": 0.45}
speed = next((v for k, v in RELATIVE.items() if k in name), 0.6)
print(f"{name}  ~{speed:.2f}x an L4")

## 2 · Install and fetch the code

In [ ]:
%%capture
!pip -q install soundfile "huggingface_hub>=0.28"
!git clone -q https://github.com/AliAkrami1375/dcho.git /content/dcho 2>/dev/null || (cd /content/dcho && git pull -q)

In [ ]:
import os, sys

sys.path.insert(0, "/content/dcho")
os.chdir("/content/dcho")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
print("signed in as", api.whoami()["name"])

!python -m unittest discover -s tests -p "test_model.py" 2>&1 | tail -3

## 3 · Configuration

`micro` is the cheap probe — use it to check that a change behaves, not to
judge audio quality. `base` is the real model.

In [ ]:
import json

CONFIG   = "micro"                  # micro | nano | base
DATASET  = "DibaAi/dcho-tier-a"     # 25,654 clips, 65.3 h, 4.6 GB packed as FLAC
RUN_REPO = "DibaAi/dcho-run-micro"  # checkpoints land here
MAX_STEPS = 30_000

# When Colab reclaims a runtime it kills the process. No exception is
# raised, no `finally` runs, nothing gets a chance to save. So the only
# state that survives is what was already uploaded, which makes the upload
# interval exactly equal to how much work a disconnect can destroy.
#
# The interval is therefore in minutes, not steps: it bounds the loss the
# same way whatever GPU this session got.
CHECKPOINT_EVERY_MINUTES = 15       # ~626 MB, fully resumable
SNAPSHOT_EVERY = 2_000              # ~20 MB, generator only, for listening

cfg = json.load(open(f"configs/{CONFIG}.json"))

steps_per_sec_l4 = {"micro": 4.0, "nano": 2.2, "base": 1.5}[CONFIG]
rate = steps_per_sec_l4 * speed
eta = MAX_STEPS / rate / 3600
print(f"{CONFIG}: {MAX_STEPS} steps, roughly {eta:.1f} h on this GPU")
print(f"  a disconnect costs at most {CHECKPOINT_EVERY_MINUTES} min "
      f"(~{int(rate * CHECKPOINT_EVERY_MINUTES * 60)} steps)")
print("  (the rate above is an estimate; real it/s appears in the log within a minute)")
if eta > 11:
    print(f"  longer than one free session — expect about {eta/11:.0f} runs of this notebook")

## 4 · Data

The packed dataset holds only the clips this phase trains on, as FLAC with
the transcript and speaker vector attached to each row. Pulling it once per
session is far cheaper than streaming the full 47 GB corpus repeatedly.

In [ ]:
import collections, time
from huggingface_hub import snapshot_download

parquet = [f for f in api.list_repo_files(DATASET, repo_type="dataset") if f.endswith(".parquet")]

# A repo can end up holding more than one packing of the same tier - a
# re-pack with different shard sizes, say. Downloading both would double the
# transfer for no benefit, so group by directory and take one complete set.
groups = collections.defaultdict(list)
for f in parquet:
    groups["/".join(f.split("/")[:-1]) or "."].append(f)
for folder, files in groups.items():
    print(f"  {folder}: {len(files)} shards")

chosen = max(groups.items(), key=lambda kv: len(kv[1]))[0]
pattern = "*.parquet" if chosen == "." else f"{chosen}/*.parquet"
print("using:", chosen)

t0 = time.time()
snapshot_download(DATASET, repo_type="dataset", local_dir="/content/data",
                  allow_patterns=[pattern, "*.json"], token=os.environ["HF_TOKEN"])
print(f"downloaded in {(time.time() - t0) / 60:.1f} min")

DATA_DIR = "/content/data" if chosen == "." else f"/content/data/{chosen}"
!du -sh {DATA_DIR}; df -h /content | tail -1

## 5 · Resume

The notebook asks the Hub whether this run already has a checkpoint. If it
does, training continues from it; if not, it starts fresh. Nothing further
down needs to know which happened.

In [ ]:
from huggingface_hub import hf_hub_download

api.create_repo(RUN_REPO, repo_type="model", private=True, exist_ok=True)

resume_from = None
try:
    files = api.list_repo_files(RUN_REPO)
    if "latest.pth" in files:
        resume_from = hf_hub_download(RUN_REPO, "latest.pth", token=os.environ["HF_TOKEN"])
        print("resuming from latest.pth")
    else:
        # Older runs numbered their checkpoints; fall back to the highest.
        numbered = [f for f in files if f.startswith("step_") and f.endswith(".pth")]
        if numbered:
            latest = max(numbered, key=lambda f: int(f[5:-4]))
            resume_from = hf_hub_download(RUN_REPO, latest, token=os.environ["HF_TOKEN"])
            print(f"resuming from {latest}")
except Exception as exc:
    print("could not check for a checkpoint:", exc)

if resume_from is None:
    print("no checkpoint found — starting from scratch")

## 6 · Build the model

In [ ]:
from dcho.model.synthesizer import Synthesizer
from dcho.text.phonemes import N_SYMBOLS
from dcho.train.trainer import Trainer

model_cfg = {k: v for k, v in cfg["model"].items() if not k.startswith("_")}
net = Synthesizer(
    n_vocab=N_SYMBOLS,
    spec_channels=cfg["data"]["filter_length"] // 2 + 1,
    segment_size=cfg["train"]["segment_size"] // cfg["data"]["hop_length"],
    n_speakers=0,
    speaker_embed_dim=192,    # continuous ECAPA vector, not a cluster id
    **model_cfg,
)
print(f"{net.n_parameters()/1e6:.2f}M parameters, "
      f"{net.n_parameters(inference_only=True)/1e6:.2f}M at inference")

# Colab bills by the hour, not per job, so the budget guard is off here and
# the session limit is what bounds the run.
cfg["train"]["max_cost_usd"] = None

trainer = Trainer(cfg, net, output_dir="/content/out", device="cuda", speaker_embed_dim=192)

if resume_from:
    trainer.load(resume_from)
    print(f"resumed at step {trainer.state.step}, lr {trainer.current_lr():.3e}")
    print(f"  {MAX_STEPS - trainer.state.step} steps remain")
else:
    print(f"fresh run, lr {trainer.current_lr():.3e}")

## 7 · Data loader

In [ ]:
from torch.utils.data import DataLoader
from dcho.data.dataset import DataConfig, PackedSpeechDataset, collate

data_cfg = DataConfig(**{k: v for k, v in cfg["data"].items()
                         if k in DataConfig.__dataclass_fields__})

# Measured at 76 clips/s per worker on this data. micro at batch 32 wants
# 4 steps/s on an L4, so two workers carry it with room to spare. On an A100
# raise num_workers or the batch size, or the GPU will sit waiting on data.
ds = PackedSpeechDataset(DATA_DIR, data_cfg)
loader = DataLoader(ds, batch_size=cfg["train"]["batch_size"], num_workers=2,
                    collate_fn=lambda b: collate(b, data_cfg),
                    pin_memory=True, persistent_workers=True, prefetch_factor=4)

batch = next(iter(loader))
print({k: tuple(v.shape) for k, v in batch.items()})
assert batch["sid"].dim() == 2, "expected continuous speaker vectors"
print("speaker vectors:", tuple(batch["sid"].shape), "norm", round(float(batch["sid"][0].norm()), 3))

## 8 · Train

Every `CHECKPOINT_EVERY` steps the weights go to the Hub. Any unexpected
exit — a tripped health guard, a manual interrupt, an unhandled error —
still writes one, so a dead session is never a wasted session.

Watch `align_H` in the log. It is the alignment entropy, and it falling
towards zero is the earliest sign the run will work at all.

In [ ]:
import time
import traceback

from dcho.train.guards import TrainingHalted

def upload(path, name):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=name,
                    repo_id=RUN_REPO, repo_type="model",
                    commit_message=f"{name} at step {trainer.state.step}")

# Always the same filename. Resume then needs no search, and the repo does
# not fill up with half-gigabyte files that will never be read again.
LATEST = "latest.pth"
last_push = time.time()

def on_log(metrics):
    global last_push
    step = metrics["step"]

    if time.time() - last_push >= CHECKPOINT_EVERY_MINUTES * 60:
        t0 = time.time()
        upload(trainer.save("latest"), LATEST)
        last_push = time.time()
        print(f"  checkpoint at step {step} (upload took {time.time()-t0:.0f}s)")

    if step % SNAPSHOT_EVERY == 0:
        upload(trainer.save_generator(f"snap_{step}"), f"snap_{step}.G.pth")

try:
    trainer.train(loader, max_steps=MAX_STEPS, checkpoint_every=10**9, on_log=on_log)
    upload(trainer.save("latest"), LATEST)
    upload(trainer.save("final"), "final.pth")
    print("finished")
except TrainingHalted as halt:
    print(f"HALTED — {halt.reason}: {halt.detail}")
    upload(trainer.save("latest"), LATEST)
except KeyboardInterrupt:
    print("interrupted; saving so the session is not wasted")
    upload(trainer.save("latest"), LATEST)
except Exception:
    traceback.print_exc()
    upload(trainer.save("latest"), LATEST)
finally:
    # A few kilobytes of loss curve, and the only thing that explains from
    # outside whether the run was healthy.
    try:
        upload(trainer.output_dir / "summary.json", "summary.json")
    except Exception as exc:
        print("could not upload the summary:", exc)

## 9 · Listen

Alignment has to lock in before anything sounds like speech. Before roughly
step 10,000 expect noise — that is normal, not a fault.

In [ ]:
from IPython.display import Audio, display

from dcho.text.frontend import Frontend

fe = Frontend()
net.eval()
speaker = batch["sid"][:1].cuda()      # reuse a real voice from the batch

for text in ["سلام، حال شما چطور است؟",
             "زبان فارسی یکی از زبان‌های کهن جهان است."]:
    out = fe(text)
    ids = torch.tensor([out.ids]).cuda()
    with torch.no_grad():
        audio, *_ = net.infer(ids, torch.tensor([ids.shape[1]]).cuda(), sid=speaker)
    print(text)
    print("  ", out.phonemes)
    display(Audio(audio.squeeze().cpu().numpy(), rate=cfg["data"]["sampling_rate"]))

---

### If the session dies

Rerun from the top. It re-downloads the data (a few minutes) and resumes
from the last checkpoint on the Hub.

### Keeping a free session alive

Leave the tab open and visible — Colab reclaims idle runtimes, and a
backgrounded tab counts as idle. Colab Pro adds background execution, which
is the only reliable way to run unattended for more than a few hours.

### Watching from outside

Every checkpoint lands in `RUN_REPO` on the Hub, so progress is visible
without touching this notebook.